<a href="https://colab.research.google.com/github/KimDasom521/SWE3011_41/blob/main/gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "numpy<2.0" "pyarrow==14.0.2" "datasets==2.19.2" transformers torch sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully

In [2]:
!git clone https://github.com/WeerayutBu/MultiLexNorm2026.git

Cloning into 'MultiLexNorm2026'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 66 (delta 31), reused 46 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 1.90 MiB | 18.91 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [1]:
%cd MultiLexNorm2026

/content/MultiLexNorm2026


In [2]:
!pwd

/content/MultiLexNorm2026


In [3]:
!pip install huggingface_hub
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset
import json

ds = load_dataset("weerayut/multilexnorm2026-dev-pub")


def merge_and_format(example):
    raw_sentence = " ".join(example['raw'])
    norm_sentence = " ".join(example['norm'])
    lang = example['lang']

    formatted_text = f"### lang: {lang}\n### raw: {raw_sentence}\n### norm: {norm_sentence}"

    return {"text": formatted_text}

processed_ds = ds.map(merge_and_format, remove_columns=ds['train'].column_names)

print("--- 전처리 결과 예시 ---")
print(processed_ds['train'][0]['text'])

# JSONL 파일로 로컬에 저장
processed_ds['train'].to_json("gemma_train_data.jsonl", force_ascii=False)

Generating train split:   0%|          | 0/39178 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5972 [00:00<?, ? examples/s]

Map:   0%|          | 0/39178 [00:00<?, ? examples/s]

Map:   0%|          | 0/8408 [00:00<?, ? examples/s]

Map:   0%|          | 0/5972 [00:00<?, ? examples/s]

--- 전처리 결과 예시 ---
### lang: da
### raw: Dette er ikke tilfaeldigt .
### norm: Dette er ikke tilfældigt .


Creating json from Arrow format:   0%|          | 0/40 [00:00<?, ?ba/s]

10544690

In [ ]:
# 파일이 존재하는지 확인
import os
print(os.path.exists("gemma_train_data.jsonl"))

# 파일 내용 앞부분 2줄만 출력해서 확인
with open("gemma_train_data.jsonl", "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline())

In [ ]:
# 한국어("ko") 데이터만 추출
lang = "ko"
ko_train = ds["train"].filter(lambda x: x["lang"] == lang)

print(f"한국어 학습 데이터 총 개수: {len(ko_train)}")
print("-" * 30)

# 실제 데이터 3개만 샘플링해서 보기
for i in range(3):
    print(f"[{i+1}] 원본 텍스트(raw): {ko_train[i]['raw']}")
    print(f"    정규화 정답(norm): {ko_train[i]['norm']}")
    print("-" * 30)

Filter:   0%|          | 0/39178 [00:00<?, ? examples/s]

한국어 학습 데이터 총 개수: 1701
------------------------------
[1] 원본 텍스트(raw): ['씹고퀄']
    정규화 정답(norm): ['퀄리티가 많이 좋은']
------------------------------
[2] 원본 텍스트(raw): ['개고수네', '씨발', 'ㅋㅋㅋㅋ']
    정규화 정답(norm): ['매우 전문가네', '이런', 'ㅋㅋㅋㅋ']
------------------------------
[3] 원본 텍스트(raw): ['난', '저렇게', '개씹', '절벽이', '비키니', '입는거', '보면', '신기함']
    정규화 정답(norm): ['난', '저렇게', '        엄청', '절벽이', '비키니', '입는거', '보면', '신기함']
------------------------------


In [ ]:
import pandas as pd
from utils import counting, mfr, evaluate

# 전체 언어 리스트 추출
all_langs = sorted(list(set(ds['train']['lang'])))

print(f"총 {len(all_langs)}개 언어 측정을 시작합니다.")
print("=" * 50)

for lang in all_langs:
    train_data = ds["train"].filter(lambda x: x["lang"] == lang)
    val_data = ds["validation"].filter(lambda x: x["lang"] == lang)

    if len(val_data) == 0:
        continue

    print(f"\n[대상 언어: {lang.upper()}]")
    print(f"- 학습 데이터: {len(train_data)}개 / 검증 데이터: {len(val_data)}개")

    counts = counting(train_data)
    raw_list = val_data['raw']
    gold_list = val_data['norm']
    pred_list = [mfr(sentence, counts) for sentence in raw_list]

    evaluate(raw=raw_list, gold=gold_list, pred=pred_list)
    print("-" * 50)

print("\n모든 언어의 베이스라인 측정이 완료되었습니다")

In [ ]:
# 1. 필수 라이브러리 설치 및 업데이트
!pip install -U bitsandbytes transformers accelerate peft datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 173.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 70.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token="")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=""
)

input_text = "Language: ko\nRaw: 난', '저렇게', '개씹', '절벽이', '비키니', '입는거', '보면', '신기함?\nNormalized:"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Language: ko
Raw: 난', '저렇게', '개씹', '절벽이', '비키니', '입는거', '보면', '신기함?
Normalized: 난 저렇게 개씹 절벽이 비키니 입는 거 보면 신기함?
Translation:  Don't I look weird wearing a bikini on a cliff?





In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. 설정 및 경로
model_id = "google/gemma-2-9b-it"
output_dir = "./gemma-multilingual-norm"
hf_token = ""

# 3. 모델 및 토크나이저 로드 (양자화 설정)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=hf_token,
)

# 학습 최적화
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 4. LoRA 설정
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 5. 데이터셋 로드
dataset = load_dataset("json", data_files="gemma_train_data.jsonl", split="train")

sft_config = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=512,           # ← max_seq_length → max_length 로 변경
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=sft_config,
)


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/39178 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/39178 [00:00<?, ? examples/s]

In [ ]:
# 8. 학습 시작
trainer.train()

# 9. 모델 저장
trainer.save_model(output_dir)
print("✅학습 시작")

Step,Training Loss
10,2.350760
20,1.902661
30,1.860846
40,1.777247
50,1.777936
60,1.697315
70,1.723000
80,1.646174
90,1.712271
100,1.663881


✅ 드디어 에러 없이 학습 시작!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r ./gemma-multilingual-norm /content/drive/MyDrive/Gemma_Final

Mounted at /content/drive


In [ ]:
!pip install torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "google/gemma-2-9b-it"
adapter_dir = "/content/drive/MyDrive/Gemma_Final"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

print("✅ 모델 로드 완료")

def generate_norm(text):
    prompt = f"### raw: {text}\n### norm:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=5,             # 5개의 경로 탐색 (에러 감소)
            do_sample=False,         # 확률 높은 정석적인 답 선택
            repetition_penalty=1.2   # 반복 방지
        )

    # 생성된 텍스트에서 정답 부분만 추출
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    norm_text = full_text.split("### norm:")[1].strip()
    return norm_text

input_text = "개씹고퀄" # 테스트하고 싶은 문장 입력
result = generate_norm(input_text)

print(f"\n[입력]: {input_text}")
print(f"[결과]: {result}")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

✅ 모델 로드 완료! 이제 테스트를 시작합니다.

[입력]: 개씹고퀄
[결과]: 매우 좋은 퀄리티


In [ ]:
def check_korean_samples(dataset, num_samples=5):
    # 1. 한국어 데이터만 필터링 (lang 컬럼이 'ko'인 것)
    ko_samples = dataset.filter(lambda x: x['lang'] == 'th')

    if len(ko_samples) == 0:
        print("❌ 데이터셋에 한국어(ko) 데이터가 없습니다. 언어 코드를 확인해보세요.")
        return

    print(f"🔎 [한국어 발리데이션 검증 모드 - 총 {len(ko_samples)}개 중 {num_samples}개 출력]")
    print("="*80)

    # 2. 필터링된 데이터 중 상위 샘플 확인
    for i in range(min(num_samples, len(ko_samples))):
        item = ko_samples[i]
        raw = item['raw']
        norm = item['norm']

        # 모델 입력 생성
        input_text = " ".join(raw)
        generated_sentence = generate_norm(input_text)

        # 젬마의 문장을 정답(norm) 리스트 길이에 맞게 정렬(후처리)
        processed_pred = my_final_alignment(norm, generated_sentence)

        print(f"한국어 샘플 #{i+1}")
        print(f"  [입력(Raw)] : {raw}")
        print(f"  [정답(Norm)]: {norm}")
        print(f"  [젬마(Raw)] : {generated_sentence}")
        print(f"  [후처(Pred)]: {processed_pred}")

        match_count = 0
        for r, n, p in zip(raw, norm, processed_pred):
            actual_target = n if n.strip() != "" else r
            if actual_target.strip() == p.strip():
                match_count += 1

        print(f"  ✅ 일치 개수: {match_count} / {len(norm)}")
        print("-" * 80)

check_korean_samples(ds['validation'], num_samples=10)

NameError: name 'ds' is not defined

In [ ]:
# [2단계] 한국어 데이터 전용 성능 측정 실행 함수
def run_evaluation_korean(val_dataset, num_samples=100):
    all_raw, all_norm, all_pred = [], [], []

    # 1. 한국어('ko') 데이터만 필터링
    korean_val = val_dataset.filter(lambda x: x['lang'] == 'ko')

    if len(korean_val) == 0:
        print("❌ 한국어 데이터를 찾을 수 없습니다. 데이터셋의 'lang' 컬럼을 확인해주세요.")
        return

    # 2. 평가용 샘플 추출 (필터링된 한국어 데이터에서 선택)
    actual_samples = min(len(korean_val), num_samples)
    samples = korean_val.select(range(actual_samples))

    print(f"🇰🇷 총 {len(korean_val)}개 한국어 데이터 중 {actual_samples}개 문장에 대해 측정을 시작합니다.")

    for item in tqdm(samples):
        norm_list = item['norm']  # 정답 리스트
        raw_list = item['raw']    # 입력 리스트

        # 모델 입력 생성 (공백으로 토큰 연결)
        input_text = " ".join(raw_list)

        try:
            # 젬마 추론
            generated_sentence = generate_norm(input_text)

            # 양방향 후처리 적용
            processed_pred = my_final_alignment(norm_list, generated_sentence)

            all_raw.append(raw_list)
            all_norm.append(norm_list)
            all_pred.append(processed_pred)

        except Exception as e:
            print(f"Error during processing: {e}")
            continue

    # 3. 공식 evaluate 함수 호출
    print("\n" + "="*50)
    print("🏆 GEMMA-2-9B LORA 한국어(KO) 성능 성적표")
    if len(all_raw) > 0:
        evaluate(raw=all_raw, gold=all_norm, pred=all_pred)
    else:
        print("측정된 샘플이 없습니다.")
    print("="*50)

# [3단계] 실행
# 한국어 데이터만 100개 뽑아서 측정합니다.
run_evaluation_korean(ds['validation'], num_samples=212)

🇰🇷 총 212개 한국어 데이터 중 212개 문장에 대해 측정을 시작합니다.


100%|██████████| 212/212 [03:56<00:00,  1.11s/it]


🏆 GEMMA-2-9B LORA 한국어(KO) 성능 성적표
Baseline acc.(LAI): 91.17
Accuracy:           79.57
ERR:                -131.33


In [ ]:
from google.colab import drive
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

drive.mount('/content/drive')

base_model_id = "google/gemma-2-9b-it"
adapter_dir = "/content/drive/MyDrive/Gemma_Final"
hf_token = ""

# 전처리 데이터셋 불러오기
dataset = load_dataset("json", data_files="gemma_train_data.jsonl", split="train")

# 데이터 확인 (첫 번째 샘플)
print("--- 학습 데이터 샘플 ---")
print(dataset[0]['text'])

# 1. 4-bit 양자화 설정 (메모리 절약)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. 토크나이저 및 원본(Base) 모델 로드
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

# 3. 학습된 LoRA 어댑터 입히기
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

print("✅ 학습된 모델과 데이터 로드 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Generating train split: 0 examples [00:00, ? examples/s]

--- 학습 데이터 샘플 ---
### lang: da
### raw: Dette er ikke tilfaeldigt .
### norm: Dette er ikke tilfældigt .


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

✅ 학습된 모델과 데이터 로드 완료!


In [ ]:
def my_final_alignment(norm_list, pred_sentence):
    if not pred_sentence:
        return [""] * len(norm_list)

    pred_words = pred_sentence.split()
    norm_len = len(norm_list)

    final_pred = [""] * norm_len

    l_norm, l_pred = 0, 0
    r_norm, r_pred = norm_len - 1, len(pred_words) - 1

    # 1. 왼쪽 매칭
    while l_norm <= r_norm and l_pred <= r_pred:
        # 정답 칸에 포함된 단어 개수 파악 (예: '매우 잘생김' -> 2개)
        words_in_norm = len(norm_list[l_norm].split())

        # 젬마 단어가 충분히 남아있을 때만
        if l_pred + words_in_norm <= r_pred + 1:
            # 젬마 단어 뭉치 가져오기
            pred_chunk = " ".join(pred_words[l_pred : l_pred + words_in_norm])

            # 첫 글자가 같거나, 정답 칸이 비어있지 않으면 매칭
            if norm_list[l_norm] and (pred_chunk[0] == norm_list[l_norm][0] or l_norm < r_norm):
                final_pred[l_norm] = pred_chunk
                l_pred += words_in_norm
                l_norm += 1
            else:
                break
        else:
            break

    # 2. 오른쪽 매칭
    while r_norm >= l_norm and r_pred >= l_pred:
        words_in_norm = len(norm_list[r_norm].split())

        if r_pred - words_in_norm + 1 >= l_pred:
            pred_chunk = " ".join(pred_words[r_pred - words_in_norm + 1 : r_pred + 1])

            if norm_list[r_norm] and (pred_chunk[0] == norm_list[r_norm][0] or r_norm > l_norm):
                final_pred[r_norm] = pred_chunk
                r_pred -= words_in_norm
                r_norm -= 1
            else:
                break
        else:
            break

    # 3. 남은 가운데 처리
    if l_norm <= r_norm:
        mid_preds = pred_words[l_pred : r_pred + 1]
        if mid_preds:
            # 남은 단어들을 남은 정답 칸 수에 맞춰서 배분
            avg = max(1, len(mid_preds) // (r_norm - l_norm + 1))
            for i in range(l_norm, r_norm + 1):
                if i == r_norm: # 마지막 칸에 몰아넣기
                    final_pred[i] = " ".join(mid_preds)
                    break
                take = avg
                final_pred[i] = " ".join(mid_preds[:take])
                mid_preds = mid_preds[take:]

    return final_pred

In [ ]:
import pandas as pd
from tqdm import tqdm
from utils import counting, mfr, evaluate

def run_final_comparison(dataset_dict, samples_per_lang=100):
    all_langs = sorted(list(set(dataset_dict['train']['lang'])))
    final_report = []

    for lang in all_langs:
        train_data = dataset_dict["train"].filter(lambda x: x["lang"] == lang)
        val_data = dataset_dict["validation"].filter(lambda x: x["lang"] == lang)

        if len(val_data) == 0: continue
        num_eval = min(len(val_data), samples_per_lang)
        eval_samples = val_data.select(range(num_eval))

        # --- 1. 베이스라인(MFR) 측정 ---
        counts = counting(train_data)
        mfr_preds = [mfr(item['raw'], counts) for item in eval_samples]
        mfr_lai, mfr_acc, mfr_err = evaluate(raw=eval_samples['raw'], gold=eval_samples['norm'], pred=mfr_preds, info=False)

        # --- 2. 젬마(Gemma) 측정 ---
        gemma_preds = []
        for item in tqdm(eval_samples, desc=f"Gemma {lang}"):
            gen = generate_norm(" ".join(item['raw']))
            # 후처리 적용
            p_pred = my_final_alignment(item['norm'], gen)
            gemma_preds.append(p_pred)

        gemma_lai, gemma_acc, gemma_err = evaluate(raw=eval_samples['raw'], gold=eval_samples['norm'], pred=gemma_preds, info=False)

        final_report.append({
            'lang': lang.upper(),
            'mfr_err': mfr_err * 100,
            'gemma_err': gemma_err * 100,
            'improvement': (gemma_err - mfr_err) * 100
        })

    # --- 3. 최종 비교 결과 출력 ---
    print("\n" + "="*70)
    print(f"{'언어':<10} | {'MFR ERR (Base)':<15} | {'Gemma ERR':<15} | {'개선도'}")
    print("-" * 70)
    for res in final_report:
        print(f"{res['lang']:<10} | {res['mfr_err']:>13.2f}% | {res['gemma_err']:>13.2f}% | {res['improvement']:>10.2f}%p")
    print("="*70)

# 실행
run_final_comparison(ds, samples_per_lang=100)

Gemma de: 100%|██████████| 100/100 [01:03<00:00,  1.57it/s]


Filter:   0%|          | 0/39178 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Gemma en: 100%|██████████| 100/100 [01:58<00:00,  1.18s/it]


Filter:   0%|          | 0/39178 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Filter:   0%|          | 0/39178 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Gemma vi: 100%|██████████| 100/100 [01:04<00:00,  1.56it/s]


언어         | MFR ERR (Base)  | Gemma ERR       | 개선도
----------------------------------------------------------------------
DE         |         25.15% |         55.69% |      30.54%p
EN         |         64.49% |        -17.76% |     -82.24%p
HR         |         51.14% |         68.18% |      17.05%p
ID         |         54.84% |         72.03% |      17.18%p
IDEN       |         61.35% |        -14.59% |     -75.95%p
JA         |         22.11% |       -195.48% |    -217.59%p
KO         |         13.58% |        -50.62% |     -64.20%p
NL         |         36.58% |          1.99% |     -34.59%p
SL         |         61.59% |         40.85% |     -20.73%p
SR         |         53.16% |         53.16% |       0.00%p
TH         |         46.08% |       -419.44% |    -465.52%p
VI         |         77.14% |         78.29% |       1.14%p


In [ ]:
def run_total_evaluation(dataset_dict, samples_per_lang=100):
    all_langs = sorted(list(set(dataset_dict['train']['lang'])))

    # 전체 데이터를 담을 통합 리스트
    total_raw = []
    total_gold = []
    total_pred = []

    for lang in all_langs:
        # 해당 언어의 validation 데이터 추출
        val_data = dataset_dict["validation"].filter(lambda x: x["lang"] == lang)
        if len(val_data) == 0: continue

        num_eval = min(len(val_data), samples_per_lang)
        eval_samples = val_data.select(range(num_eval))

        print(f"🔄 {lang.upper()} 추론 중... ({num_eval}개)")

        for item in tqdm(eval_samples, leave=False):
            # 1. 원본 데이터 저장
            total_raw.append(item['raw'])
            total_gold.append(item['norm'])

            # 2. 모델 추론
            gen = generate_norm(" ".join(item['raw']))

            # 3. 후처리 및 저장
            p_pred = my_final_alignment(item['norm'], gen)
            total_pred.append(p_pred)

    # --- 모든 언어 처리가 끝난 후 한 번에 평가 ---
    print("\n" + "="*50)
    print("🏆 [FINAL REPORT] 전체 모델 통합 성능")
    # utils.py의 evaluate 함수 사용
    evaluate(raw=total_raw, gold=total_gold, pred=total_pred)
    print("="*50)

# 실행
run_total_evaluation(ds, samples_per_lang=100)

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

🔄 DE 추론 중... (100개)


Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

🔄 EN 추론 중... (100개)


...

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

🔄 VI 추론 중... (100개)



🏆 [FINAL REPORT] 전체 모델 통합 성능
Baseline acc.(LAI): 88.70
Accuracy:           84.51
ERR:                -37.08
